# Gene Ontology

Inspect annotation evidence and ontology structure independently for Biological Process, Molecular Function and Cellular Component. Binary and information-content representations remain separate.

**Default sample: E7.5, replicate 1 — `GSM9046243_Embryo_E7.5_stereo_rep1.h5ad`.** Read only its ordered gene identifiers from `/var/_index`. Expression values, variable-gene flags and scaled-data flags do not contribute. Synthetic fallback is disabled; official mouse GO resources and identifier reports are cached through the resource-preparation step below.

Use a kernel installed from the root `requirements.txt`. This notebook uses only expression-independent sources. It does not use expression measurements. The inspection clustermap combines GO term features across the three aspects; separate block exports remain available. See [the implementation README](../README.md) for complete file schemas and audit requirements.


## Setup and parameters

All ten notebooks use the first E7.5 sample from the supplied `GSE278603_RAW` directory. Its full original gene index defines the shared ordered input list, including any duplicate rows. No expression-based filtering or ranking is performed. `GENE_ID_COLUMN=None` reads `var_names`; an explicit alternative must be a string identifier field.

The default `priors.go.json` contains the reviewed mouse GO configuration. With `PREPARE_GO_RESOURCES=True`, `prepare_go` downloads missing files from the official GO release bundle and MGI identifier reports, caches them, builds the native mouse mapping and writes the audited config. This resource bootstrap is explicit through `PREPARE_GO_RESOURCES`; `ALLOW_DOWNLOADS=False` still prevents arbitrary download descriptors during construction. To use a custom reviewed configuration, set `CONFIG_PATH` and `PREPARE_GO_RESOURCES=False`. Missing files or models are reported; synthetic data are never substituted for this sample. Inference and remote downloads require explicit opt-in.


Naming is audited before feature construction: exact native IDs and current mouse symbols take priority over synonyms; historical Ensembl gene names are retained as aliases, and unique case variants are labeled explicitly. Ensembl version suffixes and approved database prefixes are normalized; punctuation in symbols is preserved. Genuine multi-gene matches stay ambiguous. Set `PREVIOUS_MAPPING_PATH` to a previous run's mapping table to compare recovered/changed matches. The audit is saved separately and does not imply that every resolved gene has usable GO evidence. Historical GRCm38 identifiers do not establish the sample's genome assembly.


In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "preprocessing/biological_priors/core.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from preprocessing.biological_priors.core import open_text
from preprocessing.biological_priors.prepare_go import prepare_go
from preprocessing.biological_priors.audit_identifiers import audit_identifiers
from preprocessing.biological_priors.notebook_support import (
    prepare_notebook, source_overview, input_preview, execute_notebook, source_report,
    block_summary, read_block, validate_exports, plot_go_clustermap,
    covered_genes, covered_go_annotations,
)
get_ipython().run_line_magic("matplotlib", "inline")


In [ ]:
FAMILY = 'go'
DATASET_DIR = Path("/insomnia001/depts/morpheus/projects/hackathon/Datasets/GSE278603_RAW")
SAMPLE_PATH = DATASET_DIR / "GSM9046243_Embryo_E7.5_stereo_rep1.h5ad"
GENE_ID_COLUMN = None  # Read var_names (/var/_index); do not use variable/scaled flags.
CONFIG_PATH = REPO_ROOT / "preprocessing/results/priors.go.json"
PREPARE_GO_RESOURCES = True  # Ensure official inputs are cached and regenerate this default config.
ALLOW_SYNTHETIC = False
ENABLE_INFERENCE = False  # Local ESM/text inference; pretrained weights must already exist.
ALLOW_DOWNLOADS = False   # Explicit HTTPS/checksum descriptors only when enabled.
OUTPUT_ROOT = REPO_ROOT / "preprocessing/results/biological_priors_notebooks"
GO_CLUSTERMAP_REPRESENTATION = "binary"  # "binary" or "ic"; do not count both representations.
GO_CLUSTERMAP_SOURCE = None  # Auto-select the only GO source, or specify its name.
MAX_CLUSTERMAP_GENES = None  # All annotated input rows; optional integer limit for a smaller view.
SHOW_MISSINGNESS_DETAILS = False  # Optional diagnostic tables.
QUERY_GENE = None  # e.g. "Sox2"; None inspects the first covered genes.
EXPORT_CLUSTERMAP = True  # Save the figure, clustered gene order, linkage and construction details.
PREVIOUS_MAPPING_PATH = None  # Optional identifier_mapping.tsv from an earlier run, for naming-change comparison.


In [ ]:
if PREPARE_GO_RESOURCES:
    prepare_go(CONFIG_PATH)

context = prepare_notebook(
    FAMILY, config_path=CONFIG_PATH, output_root=OUTPUT_ROOT,
    sample_path=SAMPLE_PATH, gene_id_column=GENE_ID_COLUMN,
    allow_synthetic=ALLOW_SYNTHETIC, enable_inference=ENABLE_INFERENCE,
    allow_downloads=ALLOW_DOWNLOADS,
)
display(Markdown(f"**Mode: {context['mode']}**"))
print("Sample:", SAMPLE_PATH)
print("Ordered gene list:", context['genes'])
print("New run directory:", context['session'])
print("Inference sources disabled:", context['disabled_sources'])
display(pd.DataFrame([context['sample_provenance']]))
ordered_genes = context['genes'].read_text().splitlines()
print(f"{len(ordered_genes):,} input gene identifiers; first 20 shown in original order:")
display(pd.DataFrame({'input_row': range(min(20, len(ordered_genes))), 'input_id': ordered_genes[:20]}))

name_audit = audit_identifiers(
    context["genes"], context["config"]["identifiers"]["path"],
    context["session"] / "identifier_audit", previous_mapping=PREVIOUS_MAPPING_PATH,
)
print("Native mouse naming matches:", name_audit["match_type_counts"])
print("Full naming audit:", context["session"] / "identifier_audit/identifier_audit.tsv")


## Resource review and construction

Use a full mouse GAF and matched GO OBO. **GOATOOLS** supplies `GafReader` for annotation parsing, `GODag` for the ontology and alternative IDs, and `get_go2ancestors(..., relationships={"part_of"})` for traversal (including `is_a`). No local OBO parser or recursive ancestor implementation is used. The installed GOATOOLS version, GAF headers and ontology release are saved in provenance. GAF 2.x files must include all 17 columns, including blank optional fields; malformed/dropped records fail validation. Propagate `is_a` and `part_of`, omit the three GO roots and obsolete terms, and exclude negated positive memberships. Default permitted codes are IDA/IPI/IMP/IGI/TAS/IC; IEP/HEP expression-pattern evidence is prohibited. Additional evidence codes require upstream review.

For each aspect, binary features contain propagated memberships. IC features multiply memberships by `sqrt(-log((count+1)/(N+1)))`, with frequencies from the permitted mapped annotation universe. Inspect the full mapping coverage: a panel-only mapping biases IC. Root-only, filtered and zero-IC rows cannot establish similarity.

Package: [GOATOOLS](https://github.com/tanghaibao/goatools). Resource: [GO annotation downloads](https://www.geneontology.org/page/downloads).

Keep distinct source declarations and evidence channels separate. Inspect the audit below before running construction; a database name alone is not an expression-independence review.


In [ ]:
display(source_overview(context))
for spec in context['config']['sources']:
    print(spec['name'], "AUDIT:")
    print(json.dumps(spec.get('audit', {}), indent=2))
for name, lines in input_preview(context).items():
    print("\nBounded source preview:", name)
    print("\n".join(lines))


## Build and report source availability

This calls the same strict pipeline as the command-line entry point. It exports each feature matrix and its own similarity separately. Missing files/packages/models and filtered/invalid sources are reported explicitly; no block is silently filled with invented values. If no source is configured for this family, no feature block is populated.


In [ ]:
context = execute_notebook(context)
display(source_report(context))
display(block_summary(context))
manifest = context['manifest'] or {}
blocks = [read_block(context, entry) for entry in manifest.get('blocks', [])]
if manifest and SHOW_MISSINGNESS_DETAILS:
    display(pd.read_csv(context['output'] / 'identifier_mapping.tsv', sep='	', keep_default_na=False))
    print("Family coverage:", manifest['families'][FAMILY])
if not blocks:
    print("No blocks were produced. Inspect the source report and resolve dependencies/audits.")


## Family-specific inspection

Inspect the family-specific evidence and construction choices below. Only successfully populated, reviewed sources produce blocks for the sample gene list.


In [ ]:
for block in blocks:
    p = block['provenance']
    print(block['entry']['name'])
    print('Ontology/annotation backend:', json.dumps(p.get('ontology_backend', {}), indent=2))
    print('Construction:', p.get('propagation', ''), p.get('weighting', 'binary'))
    print('GO features of covered sample genes (bounded per-gene term preview):')
    display(covered_go_annotations(block, query=QUERY_GENE))
    if SHOW_MISSINGNESS_DETAILS:
        display(pd.DataFrame(p.get('excluded_annotations', [])).head(12))


## One GO gene–gene clustermap

This single view uses propagated GO term memberships across **BP, MF and CC**. Choose `GO_CLUSTERMAP_REPRESENTATION="binary"` (default) or `"ic"`; the two representations are alternatives. Term features from the three aspects are concatenated for this inspection view, and cosine similarity gives one gene–gene matrix. There is no per-aspect rescaling: each binary term contributes equally; IC uses the existing term weights.

[SciPy average linkage](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html) clusters the distances `1 - cosine similarity`. [Seaborn clustermap](https://seaborn.pydata.org/generated/seaborn.clustermap.html) uses the same tree on both axes, preserving a symmetric view. Colors show similarity from 0 to 1.

By default **every input row with any usable GO features** is plotted, including duplicate identifiers. Rows without usable annotations are excluded and counted; they cannot establish GO relationships. `MAX_CLUSTERMAP_GENES=None` means no truncation. Set an integer explicitly for a smaller view using the first annotated rows in original input order. Full clustering uses quadratic memory; labels are hidden above 60 genes, and the exported TSV supplies the full clustered gene order.

`EXPORT_CLUSTERMAP=True` saves one PNG, the clustered gene table, linkage and construction details under the run's `go_clustermap/` directory. The pipeline's separate BP/MF/CC binary/IC exports and validations remain available.


In [ ]:
# Reload the updated helper so this cell also works in an existing notebook kernel.
import importlib
from preprocessing.biological_priors import notebook_support
importlib.reload(notebook_support)

go_clustermap = notebook_support.plot_go_clustermap(
    blocks,
    representation=globals().get('GO_CLUSTERMAP_REPRESENTATION', 'binary'),
    source=globals().get('GO_CLUSTERMAP_SOURCE'),
    max_genes=globals().get('MAX_CLUSTERMAP_GENES'),
    output_dir=(context['session'] / 'go_clustermap'
                if globals().get('EXPORT_CLUSTERMAP', True) else None),
)


## Validation and exported artifacts

The pipeline checks dimensions and missing-information conventions. PSD follows from the explicit Gram construction; for large panels it records deterministic principal-submatrix eigenvalue and full-kernel quadratic-form checks rather than claiming to compute the full minimum eigenvalue. The following cell independently checks saved row alignment, labels, finite/symmetric matrices, zero missing rows and artifact hashes, and reports the recorded numerical PSD result. The checks apply to each block separately.

Source versions/URLs, upstream audits, evidence filters, construction parameters, model revisions and input hashes are retained in the manifest and each block's provenance. These biological kernels are not calibrated expression covariance. The pipeline exports and validates the original individual blocks; the combined GO clustermap is an inspection view.


In [ ]:
display(validate_exports(context))
if manifest:
    display(pd.DataFrame([{'block': entry['name'], **entry['files']} for entry in manifest['blocks']]))
    print("Manifest:", context['output'] / 'manifest.json')
print("Run/dependency report:", context['session'] / 'notebook-report.json')
print("Mode:", context['mode'])
if context['mode'] == 'SYNTHETIC DEMONSTRATION':
    print("Real biological coverage remains unassessed; no neural inference was performed.")
